Cell 1 — Imports

In [1]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
print('Imports done.')

Imports done.


Cell 2 — Load and Verify All PKL Files

In [2]:
PKL_FILES = {
    'grade_predictor'  : '../models/grade_predictor.pkl',
    'feature_scaler'   : '../models/feature_scaler.pkl',
    'feature_list'     : '../models/feature_list.pkl',
}

print('── PKL File Verification ──')
print()

all_ok = True
loaded = {}

for name, path in PKL_FILES.items():
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1024 if exists else 0
    status = '✅' if exists else '❌ MISSING'
    print(f'  {status}  {name:<25} {size:>8.1f} KB   {path}')
    if not exists:
        all_ok = False
    else:
        loaded[name] = joblib.load(path)

print()
print(f'  All files present : {"✅ YES" if all_ok else "❌ NO — fix before continuing"}')
print()

# ── Inspect contents ──────────────────────────────────────────────────────────
print('── Loaded Object Types ──')
for name, obj in loaded.items():
    print(f'  {name:<25}: {type(obj).__name__}')

print()
print('── Feature List (must be exactly 7 in correct order) ──')
for i, f in enumerate(loaded['feature_list'], 1):
    print(f'  {i}. {f}')

── PKL File Verification ──

  ✅  grade_predictor              811.5 KB   ../models/grade_predictor.pkl
  ✅  feature_scaler                 1.1 KB   ../models/feature_scaler.pkl
  ✅  feature_list                   0.1 KB   ../models/feature_list.pkl

  All files present : ✅ YES

── Loaded Object Types ──
  grade_predictor          : XGBRegressor
  feature_scaler           : StandardScaler
  feature_list             : list

── Feature List (must be exactly 7 in correct order) ──
  1. attendance_percentage
  2. midterm_score
  3. historical_gpa
  4. study_hours_per_week
  5. subject_difficulty_score
  6. ca_avg
  7. rule_risk_score


Cell 3 — Verify Feature Order Consistency

In [3]:
feature_list    = loaded['feature_list']
scaler          = loaded['feature_scaler']
regressor       = loaded['grade_predictor']

print('── Feature Order Consistency Check ──')
print()

# Scaler stores mean for each feature in the order it was fitted
scaler_feature_count = len(scaler.mean_)
print(f'  Features in feature_list.pkl : {len(feature_list)}')
print(f'  Features scaler was fitted on: {scaler_feature_count}')
print(f'  Match : {"✅ YES" if len(feature_list) == scaler_feature_count else "❌ MISMATCH — critical error"}')
print()

# Print scaler means alongside feature names for manual verification
print('── Scaler Mean per Feature (verify these match notebook 06 output) ──')
print(f"  {'Feature':<30} {'Scaler Mean':>12} {'Scaler Std':>12}")
print('  ' + '-' * 56)
for feat, mean, std in zip(feature_list, scaler.mean_, scaler.scale_):
    print(f'  {feat:<30} {mean:>12.4f} {std:>12.4f}')

print()
print(f'  XGBoost n_features_in_ : {regressor.n_features_in_}')
print(f'  Feature list length    : {len(feature_list)}')
print(f'  Match : {"✅ YES" if regressor.n_features_in_ == len(feature_list) else "❌ MISMATCH"}')

── Feature Order Consistency Check ──

  Features in feature_list.pkl : 7
  Features scaler was fitted on: 7
  Match : ✅ YES

── Scaler Mean per Feature (verify these match notebook 06 output) ──
  Feature                         Scaler Mean   Scaler Std
  --------------------------------------------------------
  attendance_percentage               82.6746      10.0941
  midterm_score                       64.2078      11.4020
  historical_gpa                       2.9570       0.4911
  study_hours_per_week                22.5177       4.4881
  subject_difficulty_score             0.5559       0.2603
  ca_avg                              64.6440       9.9813
  rule_risk_score                      0.2827       0.0777

  XGBoost n_features_in_ : 7
  Feature list length    : 7
  Match : ✅ YES


Cell 4 — Define Inference Functions


In [4]:
# These are the exact functions that will go into predictor.py
# Running them here first to verify before writing the file

def derive_grade(score: float) -> str:
    """
    Map predicted score to letter grade.
    Custom grading scale — SPAS system.
    """
    if score >= 90:   return 'A+'
    elif score >= 80: return 'A'
    elif score >= 70: return 'B+'
    elif score >= 60: return 'B'
    elif score >= 50: return 'C+'
    elif score >= 40: return 'C'
    elif score >= 30: return 'D+'
    elif score >= 20: return 'D'
    else:             return 'E'


def derive_failure_probability(score: float) -> float:
    """
    Soft failure probability derived from distance to 60-point boundary.
    Score 60  → prob 0.50 (right on boundary)
    Score 40  → prob 1.00 (very likely to fail)
    Score 80  → prob 0.00 (very unlikely to fail)
    """
    prob = (60 - score) / 20 + 0.5
    return round(float(np.clip(prob, 0.0, 1.0)), 4)


def derive_risk_level(failure_prob: float) -> str:
    """
    Map failure probability to risk level.
    Thresholds from Section 8.10 of SPAS documentation.
    """
    if failure_prob >= 0.75:   return 'CRITICAL'
    elif failure_prob >= 0.55: return 'HIGH'
    elif failure_prob >= 0.35: return 'MEDIUM'
    else:                      return 'LOW'


def predict_student(features: dict) -> dict:
    """
    Full inference pipeline for a single student.

    Args:
        features: dict with keys matching feature_list.pkl order

    Returns:
        dict with predicted_score, predicted_grade,
              failure_probability, risk_level, pass_fail
    """
    # Build feature vector in correct order
    X_input  = np.array([[features.get(f, 0) for f in feature_list]])

    # Scale
    X_scaled = scaler.transform(X_input)

    # Predict score
    predicted_score = float(np.clip(
        regressor.predict(X_scaled)[0], 0, 100
    ))

    # Derive all other outputs
    failure_prob    = derive_failure_probability(predicted_score)
    predicted_grade = derive_grade(predicted_score)
    risk_level      = derive_risk_level(failure_prob)
    pass_fail       = 1 if predicted_score >= 60 else 0

    return {
        'predicted_score'    : round(predicted_score, 2),
        'predicted_grade'    : predicted_grade,
        'failure_probability': failure_prob,
        'risk_level'         : risk_level,
        'pass_fail'          : pass_fail,
    }


print('Inference functions defined. ✅')

Inference functions defined. ✅


Cell 5 — Full Inference Test on 5 Diverse Students

In [5]:
test_students = {
    'Critical Risk Student': {
        'attendance_percentage'   : 67.0,
        'midterm_score'           : 42.0,
        'historical_gpa'          : 1.6,
        'study_hours_per_week'    : 9.0,
        'subject_difficulty_score': 0.75,
        'ca_avg'                  : 44.0,
        'rule_risk_score'         : 0.55,
    },
    'High Risk Student': {
        'attendance_percentage'   : 72.0,
        'midterm_score'           : 51.0,
        'historical_gpa'          : 2.1,
        'study_hours_per_week'    : 11.0,
        'subject_difficulty_score': 0.68,
        'ca_avg'                  : 53.0,
        'rule_risk_score'         : 0.44,
    },
    'Borderline Student': {
        'attendance_percentage'   : 76.0,
        'midterm_score'           : 59.0,
        'historical_gpa'          : 2.4,
        'study_hours_per_week'    : 13.0,
        'subject_difficulty_score': 0.60,
        'ca_avg'                  : 58.0,
        'rule_risk_score'         : 0.35,
    },
    'Medium Risk Student': {
        'attendance_percentage'   : 80.0,
        'midterm_score'           : 63.0,
        'historical_gpa'          : 2.7,
        'study_hours_per_week'    : 15.0,
        'subject_difficulty_score': 0.55,
        'ca_avg'                  : 62.0,
        'rule_risk_score'         : 0.28,
    },
    'Low Risk Student': {
        'attendance_percentage'   : 91.0,
        'midterm_score'           : 78.0,
        'historical_gpa'          : 3.5,
        'study_hours_per_week'    : 22.0,
        'subject_difficulty_score': 0.50,
        'ca_avg'                  : 76.0,
        'rule_risk_score'         : 0.14,
    },
}

print('── Full Inference Test — 5 Diverse Students ──')
print()
print(f"{'Student':<25} {'Score':>7} {'Grade':>6} "
      f"{'Fail Prob':>10} {'Risk':>10} {'Pass/Fail':>10}")
print('-' * 73)

for label, features in test_students.items():
    result = predict_student(features)
    pf_label = 'PASS' if result['pass_fail'] == 1 else 'FAIL'
    print(
        f"{label:<25} "
        f"{result['predicted_score']:>7.2f} "
        f"{result['predicted_grade']:>6} "
        f"{result['failure_probability']:>10.4f} "
        f"{result['risk_level']:>10} "
        f"{pf_label:>10}"
    )

── Full Inference Test — 5 Diverse Students ──

Student                     Score  Grade  Fail Prob       Risk  Pass/Fail
-------------------------------------------------------------------------
Critical Risk Student       48.79      C     1.0000   CRITICAL       FAIL
High Risk Student           52.40     C+     0.8798   CRITICAL       FAIL
Borderline Student          60.62      B     0.4688     MEDIUM       PASS
Medium Risk Student         63.33      B     0.3336        LOW       PASS
Low Risk Student            75.86     B+     0.0000        LOW       PASS


Cell 6 — Write predictor.py

In [ ]:
predictor_code = '''"""
predictor.py
============
SPAS Machine Learning Inference Module

This module is loaded once at FastAPI startup via initialize_models().
All prediction requests are handled by predict_student().

Usage in FastAPI:
    from app.ml.predictor import initialize_models, predict_student

    @app.on_event("startup")
    def startup():
        initialize_models()

    @router.post("/predict")
    def predict(data: PredictionInput):
        features = data.dict()
        return predict_student(features)
"""

import numpy as np
import joblib
import os

# ── Model registry — loaded once at startup ────────────────────────────────
_models = {}


def initialize_models(models_dir: str = None):
    """
    Load all PKL files into memory.
    Called once at FastAPI application startup.

    Args:
        models_dir: path to models folder.
                    Defaults to the models/ folder
                    relative to this file.
    """
    if models_dir is None:
        models_dir = os.path.join(os.path.dirname(__file__), "models")

    _models["regressor"]     = joblib.load(
        os.path.join(models_dir, "grade_predictor.pkl")
    )
    _models["scaler"]        = joblib.load(
        os.path.join(models_dir, "feature_scaler.pkl")
    )
    _models["feature_list"]  = joblib.load(
        os.path.join(models_dir, "feature_list.pkl")
    )
    print(f"[SPAS ML] Models loaded successfully from {models_dir}")
    print(f"[SPAS ML] Features : {_models['feature_list']}")


# ── Derivation helpers ─────────────────────────────────────────────────────

def derive_grade(score: float) -> str:
    """
    Map predicted score to letter grade.
    Custom grading scale — SPAS system.
    """
    if score >= 90:   return "A+"
    elif score >= 80: return "A"
    elif score >= 70: return "B+"
    elif score >= 60: return "B"
    elif score >= 50: return "C+"
    elif score >= 40: return "C"
    elif score >= 30: return "D+"
    elif score >= 20: return "D"
    else:             return "E"


def derive_failure_probability(score: float) -> float:
    """
    Soft failure probability derived from distance to
    the 60-point pass/fail boundary.

    Score = 60  -> prob = 0.50  (right on the boundary)
    Score = 40  -> prob = 1.00  (very likely to fail)
    Score = 80  -> prob = 0.00  (very unlikely to fail)
    """
    prob = (60.0 - score) / 20.0 + 0.5
    return round(float(np.clip(prob, 0.0, 1.0)), 4)


def derive_risk_level(failure_prob: float) -> str:
    """
    Map failure probability to risk level.
    Thresholds from Section 8.10 of SPAS documentation.
    """
    if failure_prob >= 0.75:   return "CRITICAL"
    elif failure_prob >= 0.55: return "HIGH"
    elif failure_prob >= 0.35: return "MEDIUM"
    else:                      return "LOW"


# ── Main inference function ────────────────────────────────────────────────

def predict_student(features: dict) -> dict:
    """
    Run full ML inference for a single student.

    Args:
        features (dict): must contain all keys in feature_list.pkl.
            Required keys:
                - attendance_percentage    (float, 0-100)
                - midterm_score            (float, 0-100)
                - historical_gpa           (float, 0.0-4.0)
                - study_hours_per_week     (float)
                - subject_difficulty_score (float, 0-1)
                - ca_avg                   (float, 0-100)
                - rule_risk_score          (float, 0-1)

    Returns:
        dict:
            - predicted_score     (float)  : predicted final exam score
            - predicted_grade     (str)    : letter grade A+/A/B+/B/C+/C/D+/D/E
            - failure_probability (float)  : probability of failing (0.0-1.0)
            - risk_level          (str)    : LOW / MEDIUM / HIGH / CRITICAL
            - pass_fail           (int)    : 1 = Pass, 0 = Fail

    Raises:
        RuntimeError: if initialize_models() has not been called yet
    """
    if not _models:
        raise RuntimeError(
            "Models not loaded. Call initialize_models() at application startup."
        )

    feature_list = _models["feature_list"]
    scaler       = _models["scaler"]
    regressor    = _models["regressor"]

    # Build feature vector in the exact order the scaler expects
    X_input  = np.array([[features.get(f, 0.0) for f in feature_list]])

    # Scale features
    X_scaled = scaler.transform(X_input)

    # Predict final score
    predicted_score = float(
        np.clip(regressor.predict(X_scaled)[0], 0.0, 100.0)
    )

    # Derive all outputs from predicted score
    failure_prob    = derive_failure_probability(predicted_score)
    predicted_grade = derive_grade(predicted_score)
    risk_level      = derive_risk_level(failure_prob)
    pass_fail       = 1 if predicted_score >= 60.0 else 0

    return {
        "predicted_score"    : round(predicted_score, 2),
        "predicted_grade"    : predicted_grade,
        "failure_probability": failure_prob,
        "risk_level"         : risk_level,
        "pass_fail"          : pass_fail,
    }
'''

# Write the file
# output_path = '../predictor.py'
# with open(output_path, 'w') as f:
#     f.write(predictor_code)

# print(f'predictor.py written to {output_path}')
# print(f'File size : {os.path.getsize(output_path) / 1024:.1f} KB')

UnicodeEncodeError: 'charmap' codec can't encode characters in position 581-582: character maps to <undefined>

Cell 7 — Test the Written predictor.py

In [8]:
# Add ml/ folder to path so we can import predictor
import importlib.util, sys

spec   = importlib.util.spec_from_file_location("predictor", "../predictor.py")
pred   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pred)

# Initialize models
pred.initialize_models(models_dir='../models')

print()
print('── Testing predictor.py with same 5 students ──')
print()
print(f"{'Student':<25} {'Score':>7} {'Grade':>6} "
      f"{'Fail Prob':>10} {'Risk':>10} {'Pass/Fail':>10}")
print('-' * 73)

for label, features in test_students.items():
    result = pred.predict_student(features)
    pf_label = 'PASS' if result['pass_fail'] == 1 else 'FAIL'
    print(
        f"{label:<25} "
        f"{result['predicted_score']:>7.2f} "
        f"{result['predicted_grade']:>6} "
        f"{result['failure_probability']:>10.4f} "
        f"{result['risk_level']:>10} "
        f"{pf_label:>10}"
    )

print()
print('✅ predictor.py works correctly and matches notebook output.')

[SPAS ML] Models loaded successfully from ../models
[SPAS ML] Features : ['attendance_percentage', 'midterm_score', 'historical_gpa', 'study_hours_per_week', 'subject_difficulty_score', 'ca_avg', 'rule_risk_score']

── Testing predictor.py with same 5 students ──

Student                     Score  Grade  Fail Prob       Risk  Pass/Fail
-------------------------------------------------------------------------
Critical Risk Student       48.79      C     1.0000   CRITICAL       FAIL
High Risk Student           52.40     C+     0.8798   CRITICAL       FAIL
Borderline Student          60.62      B     0.4688     MEDIUM       PASS
Medium Risk Student         63.33      B     0.3336        LOW       PASS
Low Risk Student            75.86     B+     0.0000        LOW       PASS

✅ predictor.py works correctly and matches notebook output.


Cell 8 — Final Pipeline Summary

In [9]:
print('╔══════════════════════════════════════════════════════════════╗')
print('║           SPAS ML PIPELINE — COMPLETE SUMMARY               ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  DATASET                                                     ║')
print('║    Source        : Synthetic demo dataset v1                 ║')
print('║    Total rows    : 5,000                                     ║')
print('║    Train / Test  : 4,000 / 1,000  (80/20 split)             ║')
print('║    Pass rate     : 76.68%                                    ║')
print('║    Fail rate     : 23.32%                                    ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  FEATURES (7 selected from 14 candidates)                   ║')
print('║    1. ca_avg                   (corr=0.8667)                 ║')
print('║    2. rule_risk_score          (corr=0.8565)                 ║')
print('║    3. midterm_score            (corr=0.8187)                 ║')
print('║    4. historical_gpa           (corr=0.6718)                 ║')
print('║    5. study_hours_per_week     (corr=0.5581)                 ║')
print('║    6. attendance_percentage    (corr=0.3175)                 ║')
print('║    7. subject_difficulty_score (corr=0.1414)                 ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  MODEL — XGBoost Regressor                                  ║')
print('║    MAE            : 2.5951  ✅ target < 8                   ║')
print('║    RMSE           : 3.2475  ✅ target < 12                  ║')
print('║    R²             : 0.8437  ✅ target ≥ 0.80                ║')
print('║    MAPE           : 4.01%                                    ║')
print('║    Exact grade    : 72.3%                                    ║')
print('║    Within 1 band  : 100.0%                                   ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  EARLY WARNING SYSTEM                                       ║')
print('║    CRITICAL precision : 98.0%                                ║')
print('║    HIGH precision     : 76.7%                                ║')
print('║    Overall recall     : 63.9%                                ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  PKL FILES READY FOR FASTAPI                                ║')
print('║    models/grade_predictor.pkl                                ║')
print('║    models/feature_scaler.pkl                                 ║')
print('║    models/feature_list.pkl                                   ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  PRODUCTION FILE                                            ║')
print('║    ml/predictor.py  ✅ written and tested                   ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  NOTEBOOKS COMPLETED                                        ║')
print('║    01 ✅ Data Generation                                     ║')
print('║    02 ✅ Data Cleaning                                       ║')
print('║    03 ✅ EDA                                                 ║')
print('║    04 ✅ Feature Engineering                                 ║')
print('║    05 ✅ Feature Selection                                   ║')
print('║    06 ✅ Model Training                                      ║')
print('║    07 ✅ Model Evaluation                                    ║')
print('║    08 ✅ Model Export                                        ║')
print('╠══════════════════════════════════════════════════════════════╣')
print('║  ML PIPELINE COMPLETE ✅                                    ║')
print('║  Next → Integrate predictor.py into FastAPI backend         ║')
print('╚══════════════════════════════════════════════════════════════╝')

╔══════════════════════════════════════════════════════════════╗
║           SPAS ML PIPELINE — COMPLETE SUMMARY               ║
╠══════════════════════════════════════════════════════════════╣
║  DATASET                                                     ║
║    Source        : Synthetic demo dataset v1                 ║
║    Total rows    : 5,000                                     ║
║    Train / Test  : 4,000 / 1,000  (80/20 split)             ║
║    Pass rate     : 76.68%                                    ║
║    Fail rate     : 23.32%                                    ║
╠══════════════════════════════════════════════════════════════╣
║  FEATURES (7 selected from 14 candidates)                   ║
║    1. ca_avg                   (corr=0.8667)                 ║
║    2. rule_risk_score          (corr=0.8565)                 ║
║    3. midterm_score            (corr=0.8187)                 ║
║    4. historical_gpa           (corr=0.6718)                 ║
║    5. study_hours_per_week